In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="lresX3mLhcAh7YJ5CZ76")
project = rf.workspace("new-workspace-ixdl4").project("vehical-detection-mssep")
version = project.version(1)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.6/324.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to vehical-detection--1 in yolov8:: 100%|██████████| 19994/19994 [00:02<00:00, 7318.88it/s]


In [ ]:
import os

In [ ]:
train_folder='/content/vehical-detection--1/train/images'
train_images=[f for f in os.listdir(train_folder) if f.lower().endswith(('jpg','jpeg','png'))]
print("lenght of trian images ....",len(train_images))
test_folder='/content/vehical-detection--1/test/images'
test_images=[f for f in os.listdir(test_folder) if f.lower().endswith(('jpg','jpeg','png'))]
print("lenght of test images is....",len(test_images))
valid_folder='/content/vehical-detection--1/valid/images'
valid_images=[f for f in os.listdir(valid_folder) if f.lower().endswith(('jpg','jpeg','png'))]
print("lenght of valid images is....",len(valid_images))

lenght of trian images .... 6993
lenght of test images is.... 1000
lenght of valid images is.... 1998


In [ ]:
from pathlib import Path
import yaml

base = Path("/content/vehical-detection--1")

# structure
print(list(base.iterdir()))

# data.yaml
d = yaml.safe_load(open(base/"data.yaml"))
print(d['names'])

# counts + classes per split
for s in ['train','valid','test']:
    imgs = list((base/s/"images").glob("*.*"))
    lbls = (base/s/"labels").glob("*.txt")
    classes = sorted({int(l.split()[0]) for f in lbls for l in open(f) if l.strip()})
    print(f"{s}: {len(imgs)} images, classes {classes}")

[PosixPath('/content/vehical-detection--1/README.roboflow.txt'), PosixPath('/content/vehical-detection--1/test'), PosixPath('/content/vehical-detection--1/valid'), PosixPath('/content/vehical-detection--1/README.dataset.txt'), PosixPath('/content/vehical-detection--1/train'), PosixPath('/content/vehical-detection--1/data.yaml')]
['auto', 'bicycle', 'bus', 'car', 'minitruck', 'motorbike', 'riksha', 'truck', 'van']
train: 6993 images, classes [0, 1, 2, 3, 4, 5, 6, 7, 8]
valid: 1998 images, classes [0, 1, 2, 3, 4, 5, 6, 7, 8]
test: 1000 images, classes [0, 1, 2, 3, 4, 5, 6, 7, 8]


In [ ]:
from collections import Counter

class_names = ['auto', 'bicycle', 'bus', 'car', 'minitruck', 'motorbike', 'riksha', 'truck', 'van']

for s in ['train', 'valid', 'test']:
    lbls = list((base/s/"labels").glob("*.txt"))

    instance_counts = Counter()   # total bounding boxes per class
    image_counts = Counter()      # images containing at least 1 of that class

    for lbl_file in lbls:
        classes_in_this_image = set()
        with open(lbl_file) as f:
            for line in f:
                if line.strip():
                    cls_id = int(line.split()[0])
                    instance_counts[cls_id] += 1
                    classes_in_this_image.add(cls_id)
        for cls_id in classes_in_this_image:
            image_counts[cls_id] += 1

    print(f"\n=== {s} ===")
    for cls_id, name in enumerate(class_names):
        print(f"{name:12s} -> images: {image_counts[cls_id]:5d} | instances: {instance_counts[cls_id]:5d}")


=== train ===
auto         -> images:  2866 | instances:  4469
bicycle      -> images:  1127 | instances:  1843
bus          -> images:   491 | instances:   615
car          -> images:  3164 | instances:  7528
minitruck    -> images:   348 | instances:   402
motorbike    -> images:  4424 | instances: 10988
riksha       -> images:   114 | instances:   122
truck        -> images:   402 | instances:   559
van          -> images:   199 | instances:   236

=== valid ===
auto         -> images:   825 | instances:  1301
bicycle      -> images:   339 | instances:   576
bus          -> images:   155 | instances:   186
car          -> images:   886 | instances:  2269
minitruck    -> images:    96 | instances:   111
motorbike    -> images:  1250 | instances:  3167
riksha       -> images:    43 | instances:    48
truck        -> images:   110 | instances:   150
van          -> images:    58 | instances:    73

=== test ===
auto         -> images:   403 | instances:   635
bicycle      -> images:  

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano — good starting point, less prone to overfitting on medium datasets

results = model.train(
    data="/content/vehical-detection--1/data.yaml",
    epochs=30,          # high ceiling — early stopping will cut it short if needed
    patience=0,         # stop if val performance doesn't improve for 15 epochs
    imgsz=640,
    batch=16,
    augment=True,        # additional YOLO-side augmentation on top of Roboflow's
    dropout=0.0,         # can increase slightly (0.1-0.2) if you still see overfitting
    val=True,# ensures validation runs every epoch so patience can track it
    project='runs/detect',
    name='stage-1'
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.166 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/vehical-detection--1/data.yaml, degrees=0.0, det

In [ ]:
import shutil
import os

# Your trained folder in Colab (change name if different)
source = '/content/runs/detect/runs/detect/stage-1'

# Where to save in Drive
destination = '/content/drive/MyDrive/YOLO_Vehicle_Project'

# Copy all files
shutil.copytree(source, destination, dirs_exist_ok=True)

print("Saved to Drive Successfully!")

Saved to Drive Successfully!
